In [2]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
#step 1 : Load all text files from the docs directory
loader = DirectoryLoader(
        path='docs',
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'}
    )

documents = loader.load()
for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")


Document 1:
  Source: docs\Google.txt
  Content length: 232201 characters
  Content preview: ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and t...
  metadata: {'source': 'docs\\Google.txt'}

Document 2:
  Source: docs\Microsoft.txt
  Content length: 201014 characters
  Content preview: ﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and ...
  metadata: {'source': 'docs\\Microsoft.txt'}


In [5]:
# step 2 : Split documents into smaller chunks with overlap
chunk_size=1000
chunk_overlap=0
text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    
chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks[:5]):
            print(f"\n--- Chunk {i+1} ---")
            print(f"Source: {chunk.metadata['source']}")
            print(f"Length: {len(chunk.page_content)} characters")
            print(f"Content:")
            print(chunk.page_content)
            print("-" * 50)

Created a chunk of size 1055, which is longer than the specified 1000
Created a chunk of size 1436, which is longer than the specified 1000
Created a chunk of size 1039, which is longer than the specified 1000
Created a chunk of size 1078, which is longer than the specified 1000
Created a chunk of size 1043, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1068, which is longer than the specified 1000
Created a chunk of size 1211, which is longer than the specified 1000
Created a chunk of size 1450, which is longer than the specified 1000
Created a chunk of size 1762, which is longer than the specified 1000
Created a chunk of size 1038, which is longer than the specified 1000
Created a chunk of size 1120, which is longer than the specified 1000
Created a chunk of size 1076, which is longer than the specified 1000
Created a chunk of size 1090, which is longer than the specified 1000
Created a chunk of s


--- Chunk 1 ---
Source: docs\Google.txt
Length: 600 characters
Content:
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--------------------------------------------------

--- Chunk 2 ---
Source: docs\Google.txt
Length: 867 characters
Content:
Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own about 14% of its publicly
listed shares and control 56% of its stockholder voting


In [9]:
# step 3 : Create and persist ChromaDB vector store
persist_directory="db1/chroma_db"

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
# Create ChromaDB vector store
print("--- Creating vector store ---")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory, 
    collection_metadata={"hnsw:space": "cosine"}
)
print("--- Finished creating vector store ---")
    
print(f"Vector store created and saved to {persist_directory}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2469.42it/s]


--- Creating vector store ---
--- Finished creating vector store ---
Vector store created and saved to db1/chroma_db
